In [33]:
from spiderstate.cat_at_origin import *
from spidercat.draw import draw_forest_on_graph
from spiderstate.utils import load_qecc

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [175]:
is_self_dual, H_x, H_z, L_x, L_z, d = load_qecc("95_1_7", "FAO")
t = d // 2
_, row_M = row_optimize_matrix(H_x, t=t, max_basis_tries=10_000)
print(row_M)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 1 0 1]
 [0 0 0 ... 0 0 1]
 ...
 [1 0 0 ... 0 0 0]
 [1 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [176]:
import numpy as np
from itertools import combinations, product
from typing import Dict, Tuple

def build_partial_lut(H: np.ndarray, L: np.ndarray, max_weight: int) -> Dict[Tuple[int, ...], int]:
    """
    Builds a partial lookup table mapping a 'Full Syndrome' (physical + logical)
    to the minimum physical weight required to generate it.
    """
    n = H.shape[1]
    lut = {}

    # Iterate in increasing order of weight to guarantee minimality
    for w in range(max_weight + 1):
        for error_indices in combinations(range(n), w):
            E = np.zeros(n, dtype=int)
            if w > 0:
                E[list(error_indices)] = 1

            # Compute physical and logical syndromes
            s = tuple((H @ E) % 2)
            l = tuple((L @ E) % 2)

            full_syndrome_key = s + l

            # Since we iterate sequentially by weight, the first time
            # we see a key, it is strictly the minimum weight for that coset.
            if full_syndrome_key not in lut:
                lut[full_syndrome_key] = w

    return lut

def find_novel_fine_hook_errors_lut(H: np.ndarray, L: np.ndarray, d: int, physical_weight: int, fault_cost: int):
    """
    Enumerates novel fine hook errors using a pre-calculated Lookup Table.
    """
    n = H.shape[1]
    k = L.shape[0]

    # We need the LUT to cover the maximum of either the hook error weight itself,
    # or the weight of the remaining faults required to break the distance.
    max_lut_weight = max(physical_weight, d - fault_cost - 1)

    # 1. Build the LUT (This is the most computationally expensive step, but only happens once)
    lut = build_partial_lut(H, L, max_lut_weight)

    # Generate all possible logical syndrome variations
    all_logical_syndromes = list(product([0, 1], repeat=k))

    for error_indices in combinations(range(n), physical_weight):
        E = np.zeros(n, dtype=int)
        E[list(error_indices)] = 1

        s_E = tuple((H @ E) % 2)
        l_E = tuple((L @ E) % 2)
        key_E = s_E + l_E

        # --- Filter 1: Coset Leader Check ---
        # If the LUT has this coset recorded at a strictly lower weight, E is degenerate.
        if lut.get(key_E, float('inf')) < physical_weight:
            continue

        # --- Filter 2: The "Fine" Check ---
        # We check all OTHER logical cosets that share the same physical syndrome.
        is_fine = True
        for l_other in all_logical_syndromes:
            if l_other == l_E:
                continue # Skip our own logical coset

            key_other = s_E + l_other

            # If an adversarial completion exists with a weight less than or equal to
            # the remaining distance budget, the hook error breaks the code.
            if lut.get(key_other, float('inf')) <= d - fault_cost - 1:
                is_fine = False
                break

        if is_fine:
            yield E

In [178]:
for x in find_novel_fine_hook_errors_lut(H=row_M, L=L_x, d=d, physical_weight=2, fault_cost=1):
    can_be_split = np.all(row_M - x >= 0, axis=1)
    if np.any(can_be_split):
        print(x)
        print(row_M[can_be_split])
        print()
    # print(np.all(M - x >= 0, axis=1))

KeyboardInterrupt: 

In [169]:
import numpy as np
from itertools import combinations, product
from typing import Dict, Tuple, List, Set

def build_partial_lut(H: np.ndarray, L: np.ndarray, max_weight: int) -> Dict[Tuple[int, ...], int]:
    """ (Same as previous implementation) """
    n = H.shape[1]
    lut = {}
    for w in range(max_weight + 1):
        for error_indices in combinations(range(n), w):
            E = np.zeros(n, dtype=int)
            if w > 0:
                E[list(error_indices)] = 1
            s = tuple((H @ E) % 2)
            l = tuple((L @ E) % 2)
            key = s + l
            if key not in lut:
                lut[key] = w
    return lut

def find_minimal_fine_hook_errors(
    H: np.ndarray,
    L: np.ndarray,
    d: int,
    weight_cost_targets: List[Tuple[int, int]]
) -> List[np.ndarray]:
    """
    Enumerates fine hook errors across multiple weights/costs, strictly filtering out
    any errors that are supersets of previously found fine errors.

    Args:
        weight_cost_targets: A list of tuples (physical_weight, fault_cost),
                             MUST be sorted by physical_weight ascending.
    """
    n = H.shape[1]
    k = L.shape[0]

    # Calculate the maximum weight needed for the LUT across all targets
    max_lut_weight = 0
    for w, c in weight_cost_targets:
        required_weight = max(w, d - c - 1)
        if required_weight > max_lut_weight:
            max_lut_weight = required_weight

    lut = build_partial_lut(H, L, max_lut_weight)
    all_logical_syndromes = list(product([0, 1], repeat=k))

    minimal_fine_errors = []
    found_error_sets: List[Set[int]] = [] # Stores indices of found minimal errors

    for physical_weight, fault_cost in weight_cost_targets:
        for error_indices in combinations(range(n), physical_weight):
            candidate_set = set(error_indices)

            # --- FILTER 1: Strict Superset Exclusion ---
            # If this candidate contains ANY previously found minimal error, discard it.
            if any(base_error.issubset(candidate_set) for base_error in found_error_sets):
                continue

            E = np.zeros(n, dtype=int)
            E[list(error_indices)] = 1

            s_E = tuple((H @ E) % 2)
            l_E = tuple((L @ E) % 2)
            key_E = s_E + l_E

            # --- FILTER 2: Coset Leader Check ---
            if lut.get(key_E, float('inf')) < physical_weight:
                continue

            # --- FILTER 3: The "Fine" Distance Check ---
            is_fine = True
            for l_other in all_logical_syndromes:
                if l_other == l_E:
                    continue

                key_other = s_E + l_other
                if lut.get(key_other, float('inf')) <= d - fault_cost - 1:
                    is_fine = False
                    break

            if is_fine:
                minimal_fine_errors.append(E)
                found_error_sets.append(candidate_set) # Save for future superset filtering

    return minimal_fine_errors

find_minimal_fine_hook_errors(H=row_M, L=L_x, d=d, weight_cost_targets=[(3, 1), (4, 1)])

[array([1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0]),
 array([1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0]),
 array([1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0]),
 array([1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0]),
 array([1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0]),
 array([1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0]),
 array([1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 0, 0, 0, 0, 0, 0, 0, 0]),
 array([1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 1, 0, 0, 0, 0, 0, 0, 0]),
 array([1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 1, 0]),
 

In [171]:
import numpy as np
from typing import List, Tuple

# ... [Assume build_partial_lut and find_minimal_fine_hook_errors are defined here as before] ...

def get_max_stabilizer_degree(H: np.ndarray) -> int:
    """Calculates the maximum number of data qubits involved in any single parity check."""
    return int(np.max(np.sum(H, axis=1)))

def auto_generate_targets(max_fault_cost: int, max_degree: int) -> List[Tuple[int, int]]:
    """
    Smartly generates (physical_weight, fault_cost) targets.
    A fault cost c can produce a physical weight between c and c * max_degree.
    """
    targets = set()

    for c in range(1, max_fault_cost):
        # A cost of c faults can result in weights from c up to c * max_degree
        for w in range(c, (c * max_degree) + 1):
            targets.add((w + 1, c))

    # CRITICAL: We must sort primarily by physical weight (ascending)
    # for the superset filter to work correctly.
    # Secondary sort by fault cost (ascending).
    sorted_targets = sorted(list(targets), key=lambda x: (x[0], x[1]))
    return sorted_targets

def auto_find_all_minimal_fine_errors(
    H: np.ndarray,
    L: np.ndarray,
    d: int,
    max_fault_cost: int
) -> List[np.ndarray]:
    """
    Autonomous wrapper that deduces the physically possible weight bounds
    from the parity matrix and executes the minimal fine error search.
    """
    # 1. Deduce the maximum possible propagation from a single fault
    max_degree = get_max_stabilizer_degree(H)

    # 2. Smartly generate the necessary targets
    targets = auto_generate_targets(max_fault_cost, max_degree)

    # 3. Execute the previously optimized subset-filtered search
    minimal_fine_errors = find_minimal_fine_hook_errors(H, L, d, targets)

    return minimal_fine_errors

auto_find_all_minimal_fine_errors(H=row_M, L=L_x, d=d, max_fault_cost=2)

KeyboardInterrupt: 